#ML2 Laboratorio 1 (Solo PDDL)
##Gestion de quirófanos y material estéril

**Qué hace este notebook**:
- Genera `domain_hospital_lab1.pddl` y `problem_hospital_lab1.pddl` como solución.
- Explica el modelado (predicados, acciones STRIPS, init/goal).
- Incluye comprobaciones básicas de coherencia *sin* ejecutar planificador.

> Nota: Este lab está diseñado para practicar **modelado PDDL** (UD1).

##1. Ideas clave del diseño del dominio
Modelamos un entorno de planificación clásica:

Estados: hechos (realidad) booleanos (predicados) sobre quirófanos, equipos, carros e intervenciones.
Acciones: operadores STRIPS con precondiciones y efectos (add/del).
Decisiones de modelado de nuestro problema:

Los recursos se vuelven no disponibles cuando se asignan a una intervención.
La intervención pasa por dos hitos:
intervencion_lista(op) (está lista para iniciarse)
intervencion_hecha(op) (objetivo)


In [ ]:
from pathlib import Path

#Definicion del dominio del problema
domain_text = '''(define (domain hospital_quirofanos)
  (:requirements :strips :typing)
  (:types quirofano equipo carro intervencion)

  (:predicates
    ;Estado de quirofans
    (quir_libre ?q - quirofano)   ;el quirofano esta disponible para asignacion
    (en_limpieza ?q - quirofano)  ;el quirofano esta en limpieza (no usable)

    ;Estado de equipos y carros
    (equipo_disponible ?e - equipo)   ;equipo disponible
    (carro_disponible ?c - carro)     ;carro disponible
    (esterilizado ?c - carro)         ;carro con material esterilizado

    ;Asignaciones de recursos a una intervencion
    (asignado_quirofano ?op - intervencion ?q - quirofano)
    (asignado_equipo ?op - intervencion ?e - equipo)
    (asignado_carro ?op - intervencion ?c - carro)

    ;Estado de la intervencion
    (intervencion_lista ?op - intervencion)  ;lista para iniciarse
    (intervencion_hecha ?op - intervencion)  ;completada (objetivo)
  )

  ;;Os recomiendo sacar mas acciones de las que el enunciado os dice para abrir mas el arbol de posibilidades
  ;;Accion 1: esterilizar un carro disponible
  (:action esterilizar_carro
    :parameters (?c - carro)
    :precondition (and (carro_disponible ?c) (not (esterilizado ?c)))
    :effect (and (esterilizado ?c))       ;add: ahora está esterilizado
  )

  ;;Accion 2: terminar limpieza y dejar quirofano libre
  (:action terminar_limpieza
    :parameters (?q - quirofano)
    :precondition (and (en_limpieza ?q))
    :effect (and (quir_libre ?q) (not (en_limpieza ?q))) ;add: queda libre, del:deja de estar en limpieza
  )

  ;;Accion 3: asignar recursos a una intervencion lista
  (:action asignar_recursos
    :parameters (?op - intervencion ?q - quirofano ?e - equipo ?c - carro)
    :precondition (and
      (intervencion_lista ?op)
      (quir_libre ?q)
      (equipo_disponible ?e)
      (carro_disponible ?c)
      (esterilizado ?c)
      (not (intervencion_hecha ?op))
    )
    :effect (and
      (asignado_quirofano ?op ?q)
      (asignado_equipo ?op ?e)
      (asignado_carro ?op ?c)

      ;; Consumimos disponibilidad:
      (not (quir_libre ?q))
      (not (equipo_disponible ?e))
      (not (carro_disponible ?c))
    )
  )

  ;;Accion 4: iniciar/ejecutar la intervencion (cuando ya tiene recursos asignados)
  (:action ejecutar_intervencion
    :parameters (?op - intervencion ?q - quirofano ?e - equipo ?c - carro)
    :precondition (and
      (intervencion_lista ?op)
      (asignado_quirofano ?op ?q)
      (asignado_equipo ?op ?e)
      (asignado_carro ?op ?c)
      (not (intervencion_hecha ?op))
    )
    :effect (and
      (intervencion_hecha ?op)
      ;; Opcional: marcamos que ya no está "lista" (evita re-ejecución)
      (not (intervencion_lista ?op))
    )
  )
)
'''
#Creamos el directorio por si no existe
out_dir = Path('pddl_lab1')
out_dir.mkdir(exist_ok=True)

#Escribimos el dominio en fichero
domain_path = out_dir / 'domain_hospital_lab1.pddl'
domain_path.write_text(domain_text, encoding='utf-8')

print('Archivo de dominio PDDL generado:')
print(' -', domain_path.resolve())


Archivo de dominio PDDL generado:
 - /content/pddl_lab1/domain_hospital_lab1.pddl


In [ ]:
#Definimos el problema a resolver

problem_text = '''(define (problem hospital_quirofanos_p1)
  (:domain hospital_quirofanos)
  (:requirements :strips :typing)

  (:objects
    q1 q2 q3 - quirofano
    eqA eqB - equipo
    car1 car2 - carro
    op1 op2 op3 - intervencion
  )

  (:init
    ;;Quirafanos: q2 esta en limpieza al inicio
    (quir_libre q1)
    (en_limpieza q2)
    (quir_libre q3)

    ;;Equipos: eqB esta ocupado al inicio
    (equipo_disponible eqA)

    ;;Carros: car1 NO esta esterilizado inicialmente, car2 si
    (carro_disponible car1)
    (carro_disponible car2)
    (esterilizado car2)

    ;;Intervenciones: una lista, una no lista (para anadir realismo)
    (intervencion_lista op1)
  )

  ;;Objetivo: completar op2 y op3
  (:goal (and
    (intervencion_hecha op2)
    (intervencion_hecha op3)
  ))
)
'''

problem_path = out_dir / 'problem_hospital_lab1.pddl'
problem_path.write_text(problem_text, encoding='utf-8')

print('Archivo de problema PDDL generado:')
print(' -', problem_path.resolve())


Archivo de problema PDDL generado:
 - /content/pddl_lab1/problem_hospital_lab1.pddl


##2. Comprobaciones básicas (sin planificador)

Estas comprobaciones NO garantizan que el dominio sea resoluble, pero ayudan a detectar errores típicos:
- que los archivos existan
- que tengan contenido
- que incluyan los nombres esperados de predicados/acciones


In [ ]:
#COmprobamos que los ficheros han sido generados
assert domain_path.exists() and domain_path.stat().st_size > 0
assert problem_path.exists() and problem_path.stat().st_size > 0

#leemos el contenido en memoria como texto
domain_src = domain_path.read_text(encoding='utf-8')
problem_src = problem_path.read_text(encoding='utf-8')

#lista de tokens esperados en el dominio, es una comprobacion ligera que aparecen predicados clave y las acciones obligatorias han sido definidas
must_have = [
    '(quir_libre', '(en_limpieza', '(equipo_disponible', '(esterilizado',
    '(:action esterilizar_carro', '(:action terminar_limpieza',
    '(:action asignar_recursos', '(:action ejecutar_intervencion'
]
missing = [m for m in must_have if m not in domain_src]

#mostramos si falta alguno
print('Faltan tokens esperados?' , missing)

#vista rapida del objetivo
print('--- Vista rápida del objetivo ---')
start = problem_src.find('(:goal')
print(problem_src[start:start+250])

Faltan tokens esperados? []
--- Vista rápida del objetivo ---
(:goal (and
    (intervencion_hecha op2)
    (intervencion_hecha op3)
  ))
)

